# 02 · Anonimización determinística (sin IA)

<a href="https://colab.research.google.com/github/manuelarguelles/tyv-demo-colab/blob/main/notebooks/02_anonimizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

Segundo paso del pipeline: antes de que **cualquier** modelo de lenguaje vea
el currículum, el sistema retira los datos que identifican a la persona
(nombre, correo, teléfono, DNI, fecha de nacimiento, dirección) y los
reemplaza por marcadores como `[CORREO_1]`.

Este notebook es **independiente** de `01`: subís vos mismo un PDF real y lo
extraemos acá (mismas 3 celdas de `01`, resumidas), para poder anonimizar
sin depender de haber corrido otro notebook antes.

**Punto clave:** esto **no lo decide un modelo de IA**. Es una
transformación 100% determinística por **expresiones regulares** en Python
— el mismo texto de entrada siempre produce exactamente la misma salida.
Eso es lo que la hace auditable.

**Límite declarado, no escondido:** anonimizar por patrones deja pasar lo
que no tiene forma de patrón reconocible. Es una capa de protección, no una
garantía absoluta de anonimato — la literatura (Parasurama y Sedoc, 2022)
muestra que el propio lenguaje de un CV puede filtrar señales como el
género incluso sin nombres.


## 1. Subir el CV real y extraer su texto

In [ ]:
!apt-get -qq update && apt-get -qq install -y poppler-utils > /dev/null


In [ ]:
def cargar_pdf() -> str:
    """Pide un PDF real al usuario. En Colab, abre el selector de archivos
    del navegador — el archivo se sube a la sesión y NO queda guardado en
    este repositorio. Corriendo localmente (fuera de Colab), busca un PDF
    ya copiado a `materiales/cvs/` (ver materiales/README.md)."""
    try:
        from google.colab import files
        print("Subí el PDF de un CV real (queda solo en esta sesión de Colab).")
        subido = files.upload()
        if not subido:
            raise RuntimeError("No se subió ningún archivo.")
        return next(iter(subido))
    except ImportError:
        import glob
        candidatos = sorted(glob.glob("materiales/cvs/*.pdf"))
        if not candidatos:
            raise FileNotFoundError(
                "Corriendo fuera de Colab: copiá un PDF real a materiales/cvs/ "
                "(ver materiales/README.md) y volvé a correr esta celda."
            )
        print(f"Usando el primer PDF encontrado en materiales/cvs/: {candidatos[0]}")
        return candidatos[0]

ruta_pdf = cargar_pdf()
print(f"\nArchivo listo: {ruta_pdf}")


In [ ]:
import subprocess

def extraer_texto_pdf(ruta_pdf: str) -> str:
    """Equivalente a la función `extraer()` del servidor real:
    `pdftotext -layout <pdf> -` conserva el orden espacial del texto."""
    resultado = subprocess.run(
        ["pdftotext", "-layout", ruta_pdf, "-"],
        capture_output=True, text=True, check=True,
    )
    return resultado.stdout

MAX_CV = 12_000  # caracteres — mismo límite que usa el sistema real (p90 sobre 287 CVs)

def recortar_a_limite(texto: str, limite: int = MAX_CV) -> str:
    if len(texto) <= limite:
        return texto
    return texto[:limite] + "\n[... recortado: documento más largo que el límite operativo ...]"

texto_extraido = recortar_a_limite(extraer_texto_pdf(ruta_pdf))
print(f"{len(texto_extraido)} caracteres extraídos.")


## 2. El nombre del candidato (para poder redactarlo)

Un nombre no tiene "forma" de patrón regex — el sistema real lo recibe del
**formulario de postulación**, no lo adivina del propio CV. Completá acá el
nombre tal como aparece en el documento que subiste (edita la celda).

In [ ]:
NOMBRE_CANDIDATO = ""  # ← editá esta línea con el nombre completo del candidato

if not NOMBRE_CANDIDATO.strip():
    print("⚠ Dejaste NOMBRE_CANDIDATO vacío: el nombre NO se podrá redactar "
          "(sí se redactarán correo/teléfono/DNI/fecha/dirección, que sí "
          "tienen forma de patrón).")


## 3. Los patrones de anonimización

Cada patrón corre **en orden**, y el orden es parte del diseño (ver comentarios en el código: por qué correo antes que teléfono/DNI, por qué nombre al final).

In [ ]:
import re
from dataclasses import dataclass
from typing import Sequence

@dataclass(frozen=True)
class Redaccion:
    tipo: str
    original: str
    marcador: str

_SEP = r"[\s.\x1f-]"

PATRONES = (
    ("correo", re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")),
    ("telefono", re.compile(rf"(?:\+?51{_SEP}{{0,3}})?9(?:{_SEP}{{0,3}}\d){{8}}(?!\d)")),
    ("telefono", re.compile(rf"\(?\s*0{_SEP}{{0,2}}1\s*\)?{_SEP}{{0,3}}\d(?:{_SEP}{{0,3}}\d){{6}}(?!\d)")),
    ("documento", re.compile(r"\b\d{8}\b")),
    ("documento", re.compile(r"\b(?:CE|C\.E\.|pasaporte)[\s:]*[A-Z0-9]{6,12}\b", re.IGNORECASE)),
    ("fecha_de_nacimiento", re.compile(
        r"\b(?:fecha\s+de\s+nacimiento|nacid[oa]\s+el|f\.?\s?nac\.?)[\s:]*\d{1,2}[/\-\s]\w{1,10}[/\-\s]\d{2,4}",
        re.IGNORECASE)),
    ("direccion", re.compile(
        r"\b(?:av\.?|avenida|jr\.?|jir[oó]n|calle|urb\.?|urbanizaci[oó]n|mz\.?|psje\.?|pasaje)\s+[^\n,;]{3,60}",
        re.IGNORECASE)),
)

def anonimizar(texto: str, nombres_conocidos: Sequence[str] = ()) -> tuple[str, list]:
    """Aplica los patrones en orden y luego los nombres conocidos (que
    vienen del formulario de postulación, NO del propio CV)."""
    redacciones, resultado, contadores = [], texto, {}
    def marcador_de(tipo):
        contadores[tipo] = contadores.get(tipo, 0) + 1
        return f"[{tipo.upper()}_{contadores[tipo]}]"
    for tipo, expresion in PATRONES:
        def reemplazo(m, _tipo=tipo):
            marcador = marcador_de(_tipo)
            redacciones.append(Redaccion(_tipo, m.group(0), marcador))
            return marcador
        resultado = expresion.sub(reemplazo, resultado)
    for nombre in [n for n in nombres_conocidos if n and n.strip()]:
        partes = [p.strip() for p in re.split(r"\s+", nombre) if len(p.strip()) >= 3]
        for aguja in sorted({nombre, *partes}, key=len, reverse=True):
            expr = re.compile(rf"\b{re.escape(aguja)}\b", re.IGNORECASE)
            def reemplazo_nombre(m):
                marcador = marcador_de("nombre")
                redacciones.append(Redaccion("nombre", m.group(0), marcador))
                return marcador
            resultado = expr.sub(reemplazo_nombre, resultado)
    return resultado, redacciones

def filtraciones_identificatorias(texto: str) -> list[str]:
    sospechas = []
    for expresion, etiqueta in (
        (re.compile(r"[\w.+-]+@[\w-]+\.\w{2,}"), "correo"),
        (re.compile(r"\b\d{8}\b"), "ocho dígitos seguidos"),
    ):
        for m in expresion.finditer(texto):
            sospechas.append(f"{etiqueta}: {m.group(0)}")
    return sospechas


## 4. Aplicarla sobre el CV real

In [ ]:
cv_protegido, redacciones = anonimizar(texto_extraido, nombres_conocidos=[NOMBRE_CANDIDATO])
print(cv_protegido)


## 5. El registro auditable de redacciones

In [ ]:
for r in redacciones:
    print(f"{r.tipo:20s} {r.marcador:14s} ← {r.original!r}")
print(f"\nTotal: {len(redacciones)} dato(s) personal(es) redactado(s).")


## 6. Verificación: ¿quedó algo sin redactar?

Un oráculo *independiente* de la propia función de anonimización — mismas formas de detección, pero para auditar el resultado, no para generar el reemplazo.

In [ ]:
fugas = filtraciones_identificatorias(cv_protegido)
print("Sin fugas detectadas." if not fugas else f"⚠ Posibles fugas: {fugas}")


## Siguiente paso

`cv_protegido` es el texto que sí puede llegar a un modelo de lenguaje. En
`03_siete_consultas_llm.ipynb` lo evaluamos contra los 7 criterios de la
rúbrica real, uno por uno.

---
*Este material es contenido educativo de apoyo a una tesis de maestría (Terry & Valdez — sistema de filtrado curricular). El PDF que subís y el nombre que ingresás quedan solo en la memoria de esta sesión de Colab — nunca se guardan en este repositorio ni se envían a ningún lado salvo, si activás el modo real, al proveedor del modelo (DeepSeek), y solo el texto ya anonimizado. Ver `materiales/README.md` para trabajar con archivos reales en disco de forma local.*
